---
title: Data Preparation
date: 2026-09-08
---


# Data Preparation

Tahap **Data Preparation** dalam kerangka kerja CRISP-DM bertujuan untuk mengubah data mentah (*raw data*) dari hasil audit *Data Understanding* menjadi dataset terbersihkan, utuh, terbebas dari *missing value* & *outlier*, serta diperkaya dengan **ekstraksi fitur deret waktu (TSFEL)** yang siap digunakan pada tahap pemodelan (*Modeling*).

Pada modul ini, alur pemrosesan data dirancang secara terstandar untuk dapat diterapkan pada seluruh variabel polutan udara (CO, NO2, SO2, CH4) selama 1 tahun penuh (**366 hari observasi** dari 30 08 2025 s/d 30 08 2026).

---
### Alur Kerja Pemrosesan Data Preparation:
1. **Pemuatan Dataset Mentah** (366 baris harian, audit awal missing values).
2. **Imputasi Missing Values** menggunakan teknik **Interpolasi Linear**.
3. **Identifikasi & Perbaikan Outlier** menggunakan metode **IQR (Interquartile Range)** dan **Winsorization Capping**.
4. **Ekstraksi Fitur Deret Waktu TSFEL** menghasilkan **68 fitur representatif** ($f_1, f_2, \dots, f_{68}$).
5. **Analisis Kemiripan Sinyal (Similarity Analysis)** menggunakan Korelasi Pearson & Cosine Similarity.
6. **Penyimpanan Dataset Terpreparasi** ke file `data_prepared.csv` & `features_tsfel.csv`.


### 1. Pemuatan dan Peninjauan Awal Dataset Mentah

Dataset mentah ditarik dari file `data_polutan_bangkalan.csv` dengan memfokuskan atribut penanggalan harian (`DATE_TIME`) dan nilai konsentrasi polutan Karbon Monoksida (`CO`).


In [4]:
import pandas as pd

xls = pd.ExcelFile(file_path)
print("Daftar sheet:", xls.sheet_names, "\n")

for name in xls.sheet_names:
    prev = pd.read_excel(xls, sheet_name=name, header=None, nrows=8)
    print(f"--- Sheet: {name} | ukuran pratinjau: {prev.shape} ---")
    print(prev.to_string(max_colwidth=25), "\n")

Daftar sheet: ['Data Harian', 'Statistik'] 

--- Sheet: Data Harian | ukuran pratinjau: (8, 9) ---
                     0             1                 2            3                4             5                 6             7                 8
0              Tanggal  NO2 (Harian)  NO2 (30-hari MA)  CO (Harian)  CO (30-hari MA)  SO2 (Harian)  SO2 (30-hari MA)  CH4 (Harian)  CH4 (30-hari MA)
1  2025-08-30 00:00:00           NaN           0.00001          NaN         0.026541           NaN          0.000007           NaN               NaN
2  2025-08-31 00:00:00           NaN           0.00001          NaN         0.026541      0.000007          0.000007           NaN       1892.824707
3  2025-09-01 00:00:00           NaN           0.00001          NaN         0.026541           NaN         -0.000033           NaN               NaN
4  2025-09-02 00:00:00           NaN           0.00001          NaN         0.026541     -0.000235         -0.000083           NaN               NaN
5  2025

In [1]:
import os
import numpy as np
import pandas as pd


def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, "myst.yml")) or os.path.exists(
            os.path.join(curr, ".git")
        ):
            return curr
        curr = os.path.dirname(curr)
    return os.path.abspath(os.getcwd())


PROJECT_ROOT = get_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
file_path = os.path.join(DATA_DIR, "data_polutan_bangkalan.xlsx")

# File sumber berisi data harian pada sheet Data Harian.
required_columns = [
    "Tanggal",
    "NO2 (Harian)",
    "CO (Harian)",
    "SO2 (Harian)",
    "CH4 (Harian)",
]
df_raw = pd.read_excel(file_path, sheet_name="Data Harian")
missing_columns = [column for column in required_columns if column not in df_raw.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

df_raw = df_raw[required_columns].copy()
df_raw = df_raw.rename(
    columns={
        "Tanggal": "DATE_TIME",
        "NO2 (Harian)": "NO2",
        "CO (Harian)": "CO",
        "SO2 (Harian)": "SO2",
        "CH4 (Harian)": "CH4",
    }
)
df_raw["DATE_TIME"] = pd.to_datetime(df_raw["DATE_TIME"], errors="coerce")
if df_raw["DATE_TIME"].isna().any():
    raise ValueError("Kolom Tanggal mengandung nilai tanggal yang tidak valid")

df_raw = df_raw.sort_values("DATE_TIME").drop_duplicates("DATE_TIME").reset_index(drop=True)
for column in ["CO", "NO2", "SO2", "CH4"]:
    df_raw[column] = pd.to_numeric(df_raw[column], errors="coerce")

co_raw = df_raw[["DATE_TIME", "CO"]].copy()

print(f"Dataset Berhasil Dimuat dari: {file_path}")
print(f"Sheet: Data Harian")
print(f"Dimensi Data: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom")
print("Missing values per polutan:")
print(df_raw[["CO", "NO2", "SO2", "CH4"]].isna().sum())
print("\n5 Baris Pertama Data Harian:")
print(df_raw.head())

Dataset Berhasil Dimuat dari: d:\PSD-B-main\PSD-B-main\data\data_polutan_bangkalan.xlsx
Sheet: Data Harian
Dimensi Data: 365 baris x 5 kolom
Missing values per polutan:
CO     264
NO2    254
SO2    177
CH4    364
dtype: int64

5 Baris Pertama Data Harian:
   DATE_TIME  NO2  CO       SO2  CH4
0 2025-08-30  NaN NaN       NaN  NaN
1 2025-08-31  NaN NaN  0.000007  NaN
2 2025-09-01  NaN NaN       NaN  NaN
3 2025-09-02  NaN NaN -0.000235  NaN
4 2025-09-03  NaN NaN       NaN  NaN


--- 

### 2. Penanganan Missing Values Menggunakan Teknik Imputasi (Interpolasi Linear)

Untuk mempertahankan kontinuitas deret waktu harian 366 hari tanpa membuang baris data tanggal yang hilang, dilakukan **Imputasi Interpolasi Linear** (`interpolate(method='linear')`).

#### Formulasi Matematika Interpolasi Linear:
$$\hat{x}_t = x_{t_a} + \frac{x_{t_b} - x_{t_a}}{t_b - t_a} (t - t_a)$$

#### Keterangan Simbol Rumus:
- $\hat{x}_t$ : Nilai taksiran konsentrasi CO ter-imputasi pada hari ke-$t$ yang hilang.
- $x_{t_a}$ : Nilai observasi valid pada titik tanggal sebelum hari yang hilang ($t_a < t$).
- $x_{t_b}$ : Nilai observasi valid pada titik tanggal sesudah hari yang hilang ($t_b > t$).
- $t_a, t_b$ : Indeks waktu observasi valid terdekat sebelum dan sesudah interval hilang.

#### Cara Membaca Rumus:
> *"Nilai CO pada hari yang hilang (\hat{x}_t) dihitung dengan mengambil nilai valid hari sebelumnya (x_{t_a}) ditambah dengan tingkat perubahan proporsional gradien antara dua hari valid terdekat dikalikan jarak selisih harinya."*

#### Logika Statistik:
Interpolasi linear membentuk garis linier kontinyu antara hari valid sebelum dan sesudah data hilang. Hal ini sangat aman untuk data polusi udara harian karena pergerakan polusi atmosferik berlangsung secara bertahap.


In [2]:
# Eksekusi Imputasi Interpolasi Linear
co_imputed = co_raw.copy()
co_imputed['CO_imputed'] = co_imputed['CO'].interpolate(method='linear').bfill().ffill()

missing_after = co_imputed['CO_imputed'].isna().sum()
print("Hasil Eksekusi Imputasi Interpolasi Linear:")
print(f"  - Missing Values Sebelum Imputasi : {co_raw['CO'].isna().sum()} hari")
print(f"  - Missing Values Setelah Imputasi : {missing_after} hari (100% Utuh)")

# Tampilkan contoh baris yang ter-imputasi
imputed_indices = co_raw[co_raw['CO'].isna()].index
print("\nContoh 5 Baris Data Hasil Imputasi:")
print(co_imputed.loc[imputed_indices[:5], ['DATE_TIME', 'CO', 'CO_imputed']])


Hasil Eksekusi Imputasi Interpolasi Linear:
  - Missing Values Sebelum Imputasi : 264 hari
  - Missing Values Setelah Imputasi : 0 hari (100% Utuh)

Contoh 5 Baris Data Hasil Imputasi:
   DATE_TIME  CO  CO_imputed
0 2025-08-30 NaN    0.026541
1 2025-08-31 NaN    0.026541
2 2025-09-01 NaN    0.026541
3 2025-09-02 NaN    0.026541
4 2025-09-03 NaN    0.026541


--- 

### 3. Identifikasi dan Penanganan Outlier (Interquartile Range Capping / Winsorization)

Untuk mendeteksi dan membatasi pencilan ekstrem tanpa membuang tanggal observasi, digunakan metode **Interquartile Range (IQR)** yang dilanjutkan dengan **Winsorization Capping**.

#### Formulasi Matematika Metode IQR:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Batas Bawah (Lower Bound)} = Q_1 - 1{,}5 \times \text{IQR}$$
$$\text{Batas Atas (Upper Bound)} = Q_3 + 1{,}5 \times \text{IQR}$$

#### Formulasi Winsorization Capping:
$$x_{\text{clean}} = \begin{cases} \text{Batas Atas}, & \text{jika } x_i > \text{Batas Atas} \\ \text{Batas Bawah}, & \text{jika } x_i < \text{Batas Bawah} \\ x_i, & \text{lainnya} \end{cases}$$

#### Keterangan Simbol Rumus:
- $Q_1$ : Kuartil Pertama (Persentil ke-25 data CO ter-imputasi).
- $Q_3$ : Kuartil Ketiga (Persentil ke-75 data CO ter-imputasi).
- $\text{IQR}$ : Jangkauan Interkuartil ($Q_3 - Q_1$).
- $1{,}5$ : Pengali standar baku Tukey untuk mendeteksi pencilan moderat.
- $x_{\text{clean}}$ : Nilai konsentrasi CO bersih setelah dibatasi (*capped*).

#### Cara Membaca Rumus:
> *"Batas toleransi outlier dihitung dengan mencari selisih antara Kuartil-3 dan Kuartil-1 (IQR). Setiap nilai CO yang melampaui Batas Atas (Q3 + 1,5 x IQR) dibatasi nilainya menjadi sama dengan Batas Atas tersebut, sehingga tidak merusak distribusi data."*


In [3]:
import numpy as np
import pandas as pd

# Data CO sudah dimuat dari data_polutan_bangkalan.xlsx.
co_imputed = co_raw.copy()
co_imputed["CO_imputed"] = (
    co_imputed["CO"].interpolate(method="linear").bfill().ffill()
)

missing_after = int(co_imputed["CO_imputed"].isna().sum())
print("Hasil Eksekusi Imputasi Interpolasi Linear:")
print(f"  - Missing Values Sebelum Imputasi : {co_raw['CO'].isna().sum()} hari")
print(f"  - Missing Values Setelah Imputasi : {missing_after} hari")

# Hitung statistik IQR dan batasi nilai ekstrem tanpa menghapus tanggal.
q1 = co_imputed["CO_imputed"].quantile(0.25)
q3 = co_imputed["CO_imputed"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

co_clean = co_imputed.copy()
co_clean["pollutant_clean"] = np.clip(
    co_clean["CO_imputed"], lower_bound, upper_bound
)
outliers_detected = co_imputed[
    (co_imputed["CO_imputed"] < lower_bound)
    | (co_imputed["CO_imputed"] > upper_bound)
]

print("Hasil Deteksi dan Perbaikan Outlier (IQR Capping):")
print(f"  - Kuartil 1 (Q1)            : {q1:.6f}")
print(f"  - Kuartil 3 (Q3)            : {q3:.6f}")
print(f"  - Interquartile Range (IQR) : {iqr:.6f}")
print(f"  - Batas Bawah               : {lower_bound:.6f}")
print(f"  - Batas Atas                : {upper_bound:.6f}")
print(f"  - Jumlah Outlier            : {len(outliers_detected)} hari")
print(f"  - Min Pollutant Clean       : {co_clean['pollutant_clean'].min():.6f}")
print(f"  - Max Pollutant Clean       : {co_clean['pollutant_clean'].max():.6f}")

Hasil Eksekusi Imputasi Interpolasi Linear:
  - Missing Values Sebelum Imputasi : 264 hari
  - Missing Values Setelah Imputasi : 0 hari
Hasil Deteksi dan Perbaikan Outlier (IQR Capping):
  - Kuartil 1 (Q1)            : 0.026541
  - Kuartil 3 (Q3)            : 0.032224
  - Interquartile Range (IQR) : 0.005683
  - Batas Bawah               : 0.018016
  - Batas Atas                : 0.040748
  - Jumlah Outlier            : 0 hari
  - Min Pollutant Clean       : 0.019760
  - Max Pollutant Clean       : 0.039356


--- 

### 4. Ekstraksi Fitur Deret Waktu Menggunakan Library TSFEL ($f_1$ s/d $f_{68}$)

Untuk mentransformasi sinyal deret waktu CO harian menjadi fitur-fitur numerik diskrit yang kaya informasi bagi algoritma machine learning, digunakan library **TSFEL (Time Series Feature Extraction Library)**.

Tabel berikut menunjukkan pemetaan **68 Fitur Representatif ($f_1$ s/d $f_{68}$)** dari domain Statistik, Temporal, dan Spektral:

| Kode Fitur | Nama Fitur TSFEL | Domain | Deskripsi Fitur |
| :---: | :--- | :---: | :--- |
| **f1** | `Absolute energy` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f2** | `Average power` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f3** | `ECDF Percentile Count_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f4** | `ECDF Percentile Count_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f5** | `ECDF Percentile_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f6** | `ECDF Percentile_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f7** | `ECDF_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f8** | `ECDF_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f9** | `ECDF_2` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f10** | `ECDF_3` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f11** | `ECDF_4` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f12** | `ECDF_5` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f13** | `ECDF_6` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f14** | `ECDF_7` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f15** | `ECDF_8` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f16** | `ECDF_9` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f17** | `Entropy` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f18** | `Histogram mode` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f19** | `Interquartile range` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f20** | `Kurtosis` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f21** | `Max` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f22** | `Mean` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f23** | `Mean absolute deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f24** | `Median` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f25** | `Median absolute deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f26** | `Min` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f27** | `Peak to peak distance` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f28** | `Root mean square` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f29** | `Skewness` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f30** | `Standard deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f31** | `Variance` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f32** | `Area under the curve` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f33** | `Autocorrelation` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f34** | `Centroid` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f35** | `Mean absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f36** | `Mean diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f37** | `Median absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f38** | `Median diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f39** | `Negative turning points` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f40** | `Neighbourhood peaks` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f41** | `Positive turning points` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f42** | `Signal distance` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f43** | `Slope` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f44** | `Sum absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f45** | `Zero crossing rate` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f46** | `Spectrogram mean coefficient_0.02Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f47** | `Spectrogram mean coefficient_0.03Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f48** | `Spectrogram mean coefficient_0.05Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f49** | `Spectrogram mean coefficient_0.06Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f50** | `Spectrogram mean coefficient_0.08Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f51** | `Spectrogram mean coefficient_0.0Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f52** | `Spectrogram mean coefficient_0.11Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f53** | `Spectrogram mean coefficient_0.13Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f54** | `Spectrogram mean coefficient_0.15Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f55** | `Spectrogram mean coefficient_0.16Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f56** | `Spectrogram mean coefficient_0.18Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f57** | `Spectrogram mean coefficient_0.19Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f58** | `Spectrogram mean coefficient_0.1Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f59** | `Spectrogram mean coefficient_0.21Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f60** | `Spectrogram mean coefficient_0.23Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f61** | `Spectrogram mean coefficient_0.24Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f62** | `Spectrogram mean coefficient_0.26Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f63** | `Spectrogram mean coefficient_0.27Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f64** | `Spectrogram mean coefficient_0.29Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f65** | `Spectrogram mean coefficient_0.31Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f66** | `Spectrogram mean coefficient_0.32Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f67** | `Spectrogram mean coefficient_0.34Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f68** | `Spectrogram mean coefficient_0.35Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |



In [4]:
import tsfel

# Ekstraksi Fitur TSFEL
cfg_stat = tsfel.get_features_by_domain('statistical')
stat_res = tsfel.time_series_features_extractor(cfg_stat, co_clean['pollutant_clean'], fs=1, verbose=0)

cfg_temp = tsfel.get_features_by_domain('temporal')
temp_res = tsfel.time_series_features_extractor(cfg_temp, co_clean['pollutant_clean'], fs=1, verbose=0)

cfg_spec = tsfel.get_features_by_domain('spectral')
spec_dfs = []
for f_name, f_params in cfg_spec['spectral'].items():
    if f_name not in ['Wavelet absolute mean', 'Wavelet energy', 'Wavelet standard deviation', 'Wavelet variance']:
        sub_cfg = {'spectral': {f_name: f_params}}
        try:
            res = tsfel.time_series_features_extractor(sub_cfg, co_clean['pollutant_clean'], fs=1, verbose=0)
            if res.columns.duplicated().any():
                res.columns = [f"{c}_{i}" for i, c in enumerate(res.columns)]
            spec_dfs.append(res)
        except Exception:
            pass

combined_features = pd.concat([stat_res, temp_res] + spec_dfs, axis=1)

seen = {}
clean_cols = []
for col in combined_features.columns:
    c_name = col.replace('0_', '')
    if c_name in seen:
        seen[c_name] += 1
        clean_cols.append(f"{c_name}_{seen[c_name]}")
    else:
        seen[c_name] = 0
        clean_cols.append(c_name)
combined_features.columns = clean_cols

# Ambil tepat 68 fitur (f1 s/d f68)
features_68 = combined_features.iloc[:, :68].copy()
features_68.columns = [f"f{i+1}" for i in range(68)]

print(f"Ekstraksi Fitur TSFEL Berhasil!")
print(f"Dimensi Matriks Fitur Extracted: {features_68.shape[0]} sampel x {features_68.shape[1]} fitur (f1 s/d f68)")
print("\nCuplikan Nilai 10 Fitur Pertama (f1 s/d f10):")
print(features_68.iloc[:, :10])


Ekstraksi Fitur TSFEL Berhasil!
Dimensi Matriks Fitur Extracted: 1 sampel x 68 fitur (f1 s/d f68)

Cuplikan Nilai 10 Fitur Pertama (f1 s/d f10):
         f1        f2    f3     f4        f5        f6       f7        f8  \
0  0.325726  0.000895  73.0  292.0  0.026157  0.032683  0.00274  0.005479   

         f9       f10  
0  0.008219  0.010959  


--- 

### 5. Analisis Kemiripan Sinyal Polutan (Similarity Analysis)

Untuk mengukur derajat asosiasi dan kemiripan bentuk gelombang deret waktu antara **Karbon Monoksida (CO)** dengan polutan lainnya (`NO2`, `SO2`, `CH4`), dilakukan **Analisis Kemiripan Sinyal** menggunakan dua metrik baku:

#### 1. Koefisien Korelasi Pearson ($r$):
$$r_{XY} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2} \cdot \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$

#### 2. Cosine Similarity ($S_C$):
$$S_C(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \cdot \sqrt{\sum_{i=1}^{n} B_i^2}}$$

#### Keterangan Simbol Rumus:
- $r_{XY}$ : Koefisien korelasi Pearson antara variabel utama polutan ($X$) dan polutan pembanding ($Y$).
- $S_C$ : Nilai kemiripan sudut kosinus antara vektor sinyal polutan dan polutan pembanding ($0 \le S_C \le 1$).
- $\mathbf{A}, \mathbf{B}$ : Vektor deret waktu konsentrasi polutan harian.

#### Cara Membaca Rumus:
> *"Cosine Similarity mengukur sudut antara dua vektor sinyal harian. Nilai S_C mendekati 1 menandakan pola pergerakan harian kedua polutan sangat seirama dan seorientasi."*


In [5]:
import numpy as np
from numpy.linalg import norm
import pandas as pd

# Gunakan data_bangkalan.xlsx yang sudah dimuat pada df_raw.
# Semua polutan disejajarkan berdasarkan tanggal yang sama.
comparison_columns = ["CO", "NO2", "SO2", "CH4"]
df_all = df_raw[["DATE_TIME"] + comparison_columns].copy()
for column in comparison_columns:
    df_all[column] = (
        pd.to_numeric(df_all[column], errors="coerce")
        .interpolate(method="linear")
        .bfill()
        .ffill()
    )

pearson_corr = df_all[comparison_columns].corr()["CO"]


def cosine_sim(a, b):
    denominator = norm(a) * norm(b)
    return float(np.dot(a, b) / denominator) if denominator else np.nan


cosine_sims = {}
co_vec = df_all["CO"].to_numpy(dtype=float)
for column in comparison_columns:
    cosine_sims[column] = cosine_sim(co_vec, df_all[column].to_numpy(dtype=float))

similarity_df = pd.DataFrame(
    {
        "Parameter Polutan": comparison_columns,
        "Pearson Correlation (r)": [pearson_corr[column] for column in comparison_columns],
        "Cosine Similarity (Sc)": [cosine_sims[column] for column in comparison_columns],
    }
)

print("Hasil Analisis Kemiripan Polutan Terhadap CO:")
print(similarity_df.to_string(index=False))

Hasil Analisis Kemiripan Polutan Terhadap CO:
Parameter Polutan  Pearson Correlation (r)  Cosine Similarity (Sc)
               CO                 1.000000                1.000000
              NO2                -0.021329                0.668772
              SO2                 0.027981               -0.138142
              CH4                      NaN                0.992183


--- 

### 6. Penyimpanan dan Integrasi Dataset Terpreparasi

Dataset polutan yang telah terbersihkan secara sempurna (0 missing values, 0 outliers) serta matriks 68 fitur TSFEL ($f_1 \dots f_{68}$) disimpan secara permanen ke dalam format CSV:
1. **`data_prepared.csv`**: Dataset harian ter-imputasi dan ter-capping (366 baris).
2. **`features_tsfel.csv`**: Matriks 68 fitur TSFEL ter-ekstraksi ($f_1$ s/d $f_{68}$).


In [6]:
# Penyimpanan dataset terpreparasi ke folder data utama
p1 = os.path.join(DATA_DIR, 'data_prepared.csv')
p2 = os.path.join(DATA_DIR, 'features_tsfel.csv')

co_clean[['DATE_TIME', 'pollutant_clean']].to_csv(p1, index=False)
features_68.to_csv(p2, index=False)

print(f"Berhasil menyimpan file ke folder data utama ({DATA_DIR}):")
print(f"  - File Prepared Data : '{p1}'")
print(f"  - File TSFEL Features: '{p2}'")

print("\nStatus Data Preparation: 100% SELESAI & SIAP UNTUK TAHAP MODELING!")

Berhasil menyimpan file ke folder data utama (d:\PSD-B-main\PSD-B-main\data):
  - File Prepared Data : 'd:\PSD-B-main\PSD-B-main\data\data_prepared.csv'
  - File TSFEL Features: 'd:\PSD-B-main\PSD-B-main\data\features_tsfel.csv'

Status Data Preparation: 100% SELESAI & SIAP UNTUK TAHAP MODELING!


In [9]:
import os
import numpy as np
import pandas as pd
import tsfel

# Gunakan workbook sumber yang sama dengan sel sebelumnya.
PROJECT_ROOT = get_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
file_path = os.path.join(DATA_DIR, "data_polutan_bangkalan.xlsx")

source = pd.read_excel(file_path, sheet_name="Data Harian")
column_map = {
    "Tanggal": "DATE_TIME",
    "NO2 (Harian)": "NO2",
    "CO (Harian)": "CO",
    "SO2 (Harian)": "SO2",
    "CH4 (Harian)": "CH4",
}
missing_columns = [column for column in column_map if column not in source.columns]
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

df_raw = source[list(column_map)].rename(columns=column_map)
df_raw["DATE_TIME"] = pd.to_datetime(df_raw["DATE_TIME"], errors="coerce")
if df_raw["DATE_TIME"].isna().any():
    raise ValueError("Kolom Tanggal mengandung nilai yang tidak valid")

pollutants = ["CO", "SO2", "CH4", "NO2"]
for pol in pollutants:
    df_raw[pol] = pd.to_numeric(df_raw[pol], errors="coerce")
df_raw = (
    df_raw.sort_values("DATE_TIME")
    .drop_duplicates("DATE_TIME")
    .reset_index(drop=True)
)

# Ekstraksi per domain mencegah konflik nama fitur pada TSFEL versi terbaru.
cfg_stat = tsfel.get_features_by_domain("statistical")
cfg_temp = tsfel.get_features_by_domain("temporal")
cfg_spec = tsfel.get_features_by_domain("spectral")


def extract_tsfel_features(signal):
    results = [
        tsfel.time_series_features_extractor(cfg_stat, signal, fs=1, verbose=0),
        tsfel.time_series_features_extractor(cfg_temp, signal, fs=1, verbose=0),
    ]
    for feature_name, feature_params in cfg_spec["spectral"].items():
        if feature_name in {
            "Wavelet absolute mean",
            "Wavelet energy",
            "Wavelet standard deviation",
            "Wavelet variance",
        }:
            continue
        try:
            result = tsfel.time_series_features_extractor(
                {"spectral": {feature_name: feature_params}},
                signal,
                fs=1,
                verbose=0,
            )
            results.append(result)
        except Exception:
            continue

    features = pd.concat(results, axis=1)
    unique_columns = []
    seen = {}
    for column in features.columns:
        base_name = column.replace("0_", "")
        count = seen.get(base_name, 0)
        unique_columns.append(base_name if count == 0 else f"{base_name}_{count}")
        seen[base_name] = count + 1
    features.columns = unique_columns

    # Kurtosis dan skewness tidak terdefinisi untuk sinyal konstan seperti CH4.
    # Ubah NaN/inf pada fitur turunan menjadi angka netral agar CSV siap dipakai.
    features = features.apply(pd.to_numeric, errors="coerce")
    features = features.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return features


for pol in pollutants:
    print(f"\n================ MENGOLAH POLUTAN: {pol} ================")
    values = df_raw[pol].interpolate(method="linear").bfill().ffill()
    if values.isna().all():
        print(f"[WARNING] {pol} tidak memiliki nilai valid. Dilewati...")
        continue

    df_pol = pd.DataFrame(
        {"DATE_TIME": df_raw["DATE_TIME"], "pollutant_clean": values}
    )
    p1 = os.path.join(DATA_DIR, f"data_prepared_{pol.lower()}.csv")
    p2 = os.path.join(DATA_DIR, f"features_tsfel_{pol.lower()}.csv")
    df_pol.to_csv(p1, index=False)

    print(f"Sedang mengekstrak fitur TSFEL untuk {pol}...")
    features_pol = extract_tsfel_features(values.to_numpy(dtype=float))
    features_pol.to_csv(p2, index=False)
    non_finite_count = int(~np.isfinite(features_pol.to_numpy()).all())

    print(f"Berhasil menyimpan file untuk {pol}:")
    print(f"  - Prepared Data : {os.path.basename(p1)} ({len(df_pol)} baris)")
    print(f"  - TSFEL Features: {os.path.basename(p2)} ({features_pol.shape[1]} fitur)")
    print(f"  - Nilai NaN/inf pada fitur: {non_finite_count}")

print("\nStatus Data Preparation semua polutan: SELESAI")


================ MENGOLAH POLUTAN: CO ================
Sedang mengekstrak fitur TSFEL untuk CO...
Berhasil menyimpan file untuk CO:
  - Prepared Data : data_prepared_co.csv (365 baris)
  - TSFEL Features: features_tsfel_co.csv (120 fitur)
  - Nilai NaN/inf pada fitur: 0

================ MENGOLAH POLUTAN: SO2 ================
Sedang mengekstrak fitur TSFEL untuk SO2...
Berhasil menyimpan file untuk SO2:
  - Prepared Data : data_prepared_so2.csv (365 baris)
  - TSFEL Features: features_tsfel_so2.csv (120 fitur)
  - Nilai NaN/inf pada fitur: 0

================ MENGOLAH POLUTAN: CH4 ================
Sedang mengekstrak fitur TSFEL untuk CH4...
Berhasil menyimpan file untuk CH4:
  - Prepared Data : data_prepared_ch4.csv (365 baris)
  - TSFEL Features: features_tsfel_ch4.csv (120 fitur)
  - Nilai NaN/inf pada fitur: 0

================ MENGOLAH POLUTAN: NO2 ================
Sedang mengekstrak fitur TSFEL untuk NO2...
Berhasil menyimpan file untuk NO2:
  - Prepared Data : data_prepared_no2.

c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tsfel\feature_extraction\features.py:535: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return scipy.stats.kurtosis(signal)
c:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tsfel\feature_extraction\features.py:554: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  return scipy.stats.skew(signal)
